# Taller · De los datos a un dashboard publicado

**Programación — ADMI 1301**

Trabajarás con las **2.000 ventas de una cadena de papelerías en 2025**. En este taller vas a:

1. **Graficar** (6 ejercicios con `groupby`, `crosstab` y `pivot_table`).
2. **Organizar** tus gráficos en una sola figura con `subplots`: tu *dashboard*.
3. **Embellecerlo con prompts** para Gemini, en el mismo Colab.
4. **Elegir tu versión favorita**, escribir tus conclusiones y **publicar tu notebook en GitHub**.

Las instrucciones completas están en la guía en Word. Aquí encuentras lo esencial de cada ejercicio y una **verificación** (`assert`) para saber si vas bien.

Integrantes:
Manuela Pardo
Luisa Espinosa
Sebastián Plata
Valeria Ramirez


## 0. Preparación
Ejecuta la celda y sube el archivo **`ventas_papelerias_2025.csv`**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

subidos = files.upload()   # selecciona ventas_papelerias_2025.csv

df = pd.read_csv("ventas_papelerias_2025.csv")
df["fecha"] = pd.to_datetime(df["fecha"])
df["mes"] = df["fecha"].dt.month        # 1 = enero ... 12 = diciembre
df.head()

**Conoce tus datos** antes de graficar: cuántas filas hay, qué columnas, qué valores tiene cada categoría.

In [ ]:
print(df.shape)
df.info()
for col in ["ciudad", "canal", "segmento", "categoria", "metodo_pago"]:
    print(col, "->", sorted(df[col].unique()))

---
# Parte A · Seis gráficos
En cada ejercicio: **(1)** prepara una tabla con `groupby`, `crosstab` o `pivot_table`, **(2)** guárdala en la variable indicada, **(3)** grafícala. Ponle **título** a cada gráfico.

### Ejercicio 1 · Top 5 de productos *(barras horizontales)*
**Contexto.** La gerente de compras debe decidir en qué productos concentrar el inventario del próximo año.

**Tarea.** Calcula los ingresos (`total`) por `producto`, quédate con los **5 mayores** y grafícalos en barras horizontales. Guarda la tabla en **`top_productos`**.

*Herramientas:* `groupby`, `sum`, `sort_values`, `head`, `.plot.barh()`

In [ ]:
# Este código grafica los 5 productos con mayores ingresos

top_productos = (df.groupby("producto")["total"].sum()
                .sort_values(ascending=False)
                .head(5))
display(top_productos)

top_productos.sort_values().plot.barh(color="#1f4e79", figsize=(8 ,4),
                                      title="Top 5 de productos por ingresos (2025)")
plt.xlabel("Ingresos ($)")
plt.ylabel("Producto")
plt.tight_layout()
plt.show()


### Ejercicio 2 · Ingresos mes a mes: Bogotá vs Medellín *(líneas)*
**Contexto.** La dirección regional quiere comparar la evolución de sus dos ciudades principales durante el año.

**Tarea.** **Selecciona** solo las ventas de Bogotá y Medellín (`isin`). Con una **tabla dinámica** pon los meses en las filas, las ciudades en las columnas y la suma de `total` como valores. Guárdala en **`ing_mes_ciudad`** y grafícala como líneas con marcadores.

*Herramientas:* filtrado con `isin`, `pivot_table`, `.plot.line(marker="o")`

In [ ]:
# Este ejercicio compara las ventas de Bogotá y Medellín

bog_med = df[df["ciudad"].isin(["Bogotá", "Medellín"])]
ing_mes_ciudad = bog_med.pivot_table(index="mes", columns="ciudad",
                                     values="total", aggfunc="sum")
display(ing_mes_ciudad)

ing_mes_ciudad.plot.line(marker="o", figsize=(8, 4),
                         title="Ingresos mensuales 2025:Bogotá vs Medellín")
plt.xlabel("Mes (1 = enero ... 12 = diciembre)")
plt.ylabel("Ingresos ($)")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

### Ejercicio 3 · ¿Por qué canal se vende más? *(gráfico de pie)*
**Contexto.** La empresa evalúa si invierte más en su tienda online o en la atención telefónica.

**Tarea.** Calcula los ingresos totales por `canal`, guárdalos en **`ing_canal`** y grafica la participación de cada canal con un gráfico de pie que muestre los **porcentajes con un decimal**.

*Herramientas:* `groupby`, `.plot.pie(autopct=...)`

In [ ]:
# Este ejercicio permite evaluar si hay mayor inversión en la venta online o en la telefónica

ing_canal = df.groupby("canal")["total"].sum()
display(ing_canal)

ing_canal.plot.pie(autopct="%1.1f%%", figsize=(5, 5),
                   title="Participación de cada canal en los ingresos")
plt.ylabel("")
plt.tight_layout()
plt.show()

### Ejercicio 4 · ¿Cada segmento compra lo mismo? *(barras apiladas)*
**Contexto.** El área comercial sospecha que los clientes corporativos compran productos distintos a los minoristas.

**Tarea.** Con `pd.crosstab` cuenta las ventas por `segmento` (filas) y `categoria` (columnas), **normalizadas por fila** y expresadas en **porcentaje** (de 0 a 100). Guarda la tabla en **`tabla_seg_cat`** y grafícala en barras **apiladas**.

*Herramientas:* `pd.crosstab(..., normalize="index")`, `.plot.bar(stacked=True)`

In [ ]:
# El ejercicio permite evaluar si se compran productos distintos a los minoristas

tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
display(tabla_seg_cat.round(1))

tabla_seg_cat.plot.bar(stacked=True, figsize=(8, 5), rot=0,
                       title="Composición de las ventas por categoría según segmento")
plt.xlabel("Segmento")
plt.ylabel("% de las ventas")
plt.legend(title="Categoría", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Ejercicio 5 · Productividad de los vendedores *(dispersión)*
**Contexto.** Recursos humanos revisa qué vendedores concretan más ventas y cuáles venden tickets más altos.

**Tarea.** Con `groupby` por `vendedor` y `ciudad`, calcula para cada vendedor el **número de ventas** (`n_ventas`) y el **valor promedio de sus ventas** (`ticket_prom`). Guarda la tabla en **`vend`** (con índice normal: usa `reset_index()`). Haz una dispersión (`n_ventas` en X, `ticket_prom` en Y) con un **color por ciudad** y leyenda.

*Herramientas:* `groupby().agg(nombre=("columna", "función"))`, `reset_index`, `plt.scatter` dentro de un `for`

In [ ]:
# Este ejercicio identifica a los mejores vendedores en términos de ventas concretadas y tickets

vend = (df.groupby(["vendedor", "ciudad"])
          .agg(n_ventas=("id_venta", "count"), ticket_prom=("total", "mean"))
          .reset_index())
display(vend)

plt.figure(figsize=(8, 5))
for ciudad, grupo in vend.groupby("ciudad"):
    plt.scatter(grupo["n_ventas"], grupo["ticket_prom"], label=ciudad, s=80)
plt.title("Productividad de los vendedores: n.º de ventas vs ticket promedio")
plt.xlabel("Número de ventas")
plt.ylabel("Ticket promedio ($)")
plt.legend(title="Ciudad")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Ejercicio 6 · ¿Qué tan grandes son las ventas de cada segmento? *(diagrama de cajas)*
**Contexto.** Para planear el segundo semestre, finanzas quiere ver cómo se distribuye el valor de las ventas por tipo de cliente.

**Tarea.** **Selecciona** solo las ventas del **segundo semestre** (`mes` de 7 a 12) y guárdalas en **`sem2`**. Haz un diagrama de cajas del `total` por `segmento`, **sin mostrar los valores atípicos** (`showfliers=False`).

*Herramientas:* filtro con `>=`, `.boxplot(column=..., by=...)`

In [ ]:
# Distribución de ventas por el tipo de cliente

sem2 = df[df["mes"] >= 7]
print("Ventas del segundo semestre:", len(sem2))

sem2.boxplot(column="total", by="segmento", showfliers=False, figsize=(7, 5))
plt.suptitle("")   # elimina el título que pandas arroja
plt.title("Valor de las ventas por segmento (2.º semestre, sin atípicos)")
plt.xlabel("Segmento")
plt.ylabel("Total de la venta ($)")
plt.tight_layout()
plt.show()

---
# Parte B · Todo en una sola figura (`subplots`)
### Ejercicio 7 · Organiza tus 6 gráficos en una cuadrícula de 2 × 3
**Contexto.** La gerente quiere ver todo en **una sola imagen** para presentarla en la junta.

**Tarea.** Crea una figura con `fig, axs = plt.subplots(2, 3, figsize=(18, 10))` y dibuja cada gráfico en su posición pasando **`ax=axs[fila, columna]`**. Agrega un título general con `fig.suptitle(...)` y termina con `plt.tight_layout()`.

**Importante:** escribe el código en la celda que empieza con `%%writefile figura_base.py` (así el programa guarda tu código en un archivo, que usarás en la Parte C). Luego ejecuta la celda siguiente (`%run -i figura_base.py`) para ver la figura.

*Pistas:* `plot.barh(ax=axs[0, 0])` · `plot.pie(...)` también acepta `ax` · para el boxplot: `boxplot(column=..., by=..., ax=...)` · para la dispersión: `axs[1, 1].scatter(...)`

In [ ]:
%%writefile figura_base.py
# Este ejercicio junta toda la información anterior en una sola imagen
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Tablas Parte A
top_productos = (df.groupby("producto")["total"].sum()
                   .sort_values(ascending=False).head(5))

bog_med = df[df["ciudad"].isin(["Bogotá", "Medellín"])]
ing_mes_ciudad = bog_med.pivot_table(index="mes", columns="ciudad",
                                     values="total", aggfunc="sum")

ing_canal = df.groupby("canal")["total"].sum()

tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100

vend = (df.groupby(["vendedor", "ciudad"])
          .agg(n_ventas=("id_venta", "count"), ticket_prom=("total", "mean"))
          .reset_index())

sem2 = df[df["mes"] >= 7]

# Asignación de colores por ciudad
colores_ciudad = {"Bogotá": "#1f4e79", "Medellín": "#e8832a", "Cali": "#3a9d5d",
                  "Barranquilla": "#f2c230", "Bucaramanga": "#7b4fa3"}

# Formatos de ejes (no notación científica)
en_millones = FuncFormatter(lambda x, pos: f"{x / 1e6:g}")
en_miles = FuncFormatter(lambda x, pos: f"{x / 1e3:g}")


fig, axs = plt.subplots(2, 3, figsize=(18, 10))

# Primer ejercicio: Top 5 productos
top_productos.sort_values().plot.barh(ax=axs[0, 0], color="#1f4e79",
                                      title="Top 5 de productos por ingresos")
axs[0, 0].xaxis.set_major_formatter(en_millones)
axs[0, 0].set_xlabel("Ingresos (millones de $)")
axs[0, 0].set_ylabel("Producto")

# Segundo ejercicio: Bogotá y Medellín
ing_mes_ciudad.plot.line(ax=axs[0, 1], marker="o",
                         color=[colores_ciudad[c] for c in ing_mes_ciudad.columns],
                         title="Ingresos mensuales: Bogotá vs Medellín")
axs[0, 1].yaxis.set_major_formatter(en_millones)
axs[0, 1].set_xlabel("Mes (1 = enero ... 12 = diciembre)")
axs[0, 1].set_ylabel("Ingresos (millones de $)")
axs[0, 1].set_xticks(range(1, 13))
axs[0, 1].legend(title="Ciudad")

# Tercer ejercicio: gráfica de pie
ing_canal.plot.pie(ax=axs[0, 2], autopct="%1.1f%%",
                   title="Participación de cada canal en los ingresos")
axs[0, 2].set_ylabel("")

# Cuarto ejercicio: diferencia con productos minoristas
tabla_seg_cat.plot.bar(ax=axs[1, 0], stacked=True, rot=0,
                       title="Categorías compradas por segmento")
axs[1, 0].set_xlabel("Segmento")
axs[1, 0].set_ylabel("% de las ventas del segmento")
axs[1, 0].legend(title="Categoría", fontsize=8, ncol=3, loc="upper center",
                 bbox_to_anchor=(0.5, -0.18), frameon=False)

# Quinto ejercicio: Dispersión de vendedores
for ciudad, grupo in vend.groupby("ciudad"):
    axs[1, 1].scatter(grupo["n_ventas"], grupo["ticket_prom"], label=ciudad,
                      s=80, color=colores_ciudad[ciudad])
axs[1, 1].yaxis.set_major_formatter(en_miles)
axs[1, 1].set_title("Vendedores: n.º de ventas vs ticket promedio")
axs[1, 1].set_xlabel("Número de ventas")
axs[1, 1].set_ylabel("Ticket promedio (miles de $)")
axs[1, 1].legend(title="Ciudad", fontsize=8)

# Sexto ejercicio: Boxplot - segundo semestre
sem2.boxplot(column="total", by="segmento", ax=axs[1, 2], showfliers=False,
             grid=False, medianprops=dict(color="#e8832a", linewidth=2.5))
axs[1, 2].yaxis.set_major_formatter(en_miles)
axs[1, 2].set_title("Valor de las ventas por segmento (2.º semestre)")
axs[1, 2].set_xlabel("Segmento")
axs[1, 2].set_ylabel("Total de la venta (miles de $)")

fig.suptitle("Ventas de la cadena de papelerías · 2025", fontsize=18)
plt.tight_layout()

In [ ]:
%run -i figura_base.py
display(fig)
plt.close(fig)

In [ ]:
# Verificación
assert len(fig.axes) >= 6, f"La figura debe tener 6 gráficos, tiene {len(fig.axes)}"
print("Ejercicio 7 correcto ✔")

---
# Parte C · Hazla ver bonita con prompts
Ahora **tú escribes el prompt y Gemini reescribe el código** de tu figura para que se vea profesional, casi como un dashboard.


### Herramientas *(ya están hechas)*
Ejecuta esta celda. Define dos funciones:

- **`embellecer(prompt, nombre)`**: envía tu prompt **junto con tu código de `figura_base.py`**, ejecuta el código nuevo que devuelve Gemini y muestra la figura. Cada versión queda guardada en `VERSIONES[nombre]`.
- **`mostrar(nombre)`**: vuelve a dibujar una versión guardada.

In [ ]:
import re

VERSIONES = {}      # aquí se guarda el código de cada versión ("v1", "v2", ...)

REGLAS_FIGURA = """

Reglas técnicas obligatorias:
- Usa solo pandas y matplotlib (y numpy si lo necesitas). No instales otras librerías.
- Los datos ya están cargados en el DataFrame `df` (columnas: id_venta, fecha, ciudad, canal, segmento, vendedor,
  producto, categoria, metodo_pago, cantidad, precio_unitario, descuento, total, mes). No cargues archivos.
- Conserva los MISMOS 6 gráficos y los mismos datos del código actual (puedes recalcular las tablas con el mismo código);
  cambia solo la presentación.
- La figura debe quedar en una variable llamada `fig` (2 filas x 3 columnas).
- NO uses plt.show() ni guardes archivos.
- Entrega solo UN bloque de código Python completo.
"""


def extraer_codigo(texto):
    """Devuelve el contenido del primer bloque de código ``` de la respuesta."""
    bloque = re.search(r"```(?:python)?\s*\n(.*?)```", texto, re.S)
    return bloque.group(1) if bloque else texto


def dibujar(codigo):
    """Ejecuta el código de una figura y la muestra."""
    import numpy as np
    plt.close("all")
    espacio = {"df": df, "pd": pd, "plt": plt, "np": np}
    exec(codigo, espacio)
    figura = espacio["fig"]
    display(figura)
    plt.close(figura)


def embellecer(prompt, nombre, modelo="gemini-3.8-flash"):
    """Gemini reescribe tu figura según tu prompt; se ejecuta, se muestra y se guarda en VERSIONES."""
    codigo_actual = open("figura_base.py", encoding="utf-8").read()
    contenido = f"{prompt}\n\nEste es mi código actual:\n```python\n{codigo_actual}\n```{REGLAS_FIGURA}"
    respuesta = client.models.generate_content(model=modelo, contents=contenido)
    codigo = extraer_codigo(respuesta.text)
    try:
        dibujar(codigo)
    except Exception as e:
        print("❌ El código de Gemini falló:", type(e).__name__, e)
        print("Copia ese error en tu siguiente prompt y pídele que lo corrija.")
        return
    VERSIONES[nombre] = codigo
    print(f"✔ Versión '{nombre}' guardada")


def mostrar(nombre):
    """Vuelve a dibujar una versión guardada."""
    dibujar(VERSIONES[nombre])

In [ ]:
# nos toco instalar esto, por que no dejaba correr el codigo sin esto
!pip install -q google-genai

from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GOOGLE_API_KEY")
)

print("Gemini conectado correctamente")

### Ejercicio 8 · Tu primer prompt de estilo
Completa las secciones de `prompt_v1`. Sé **específico**: "usa una paleta de azules y naranja" funciona mejor que "hazlo bonito".

- **PÚBLICO Y OBJETIVO:** quién verá la figura y para qué.
- **PALETA Y TIPOGRAFÍA:** colores (por ciudad, por producto…), fondo, tamaños de letra.
- **TÍTULOS:** que cada gráfico tenga un título que diga la **conclusión** (por ejemplo, "Bogotá lidera las ventas"), no solo el nombre.
- **EJES Y NÚMEROS:** formatos con separador de miles o "$ 12 M", etiquetas de ejes claras.
- **LIMPIEZA VISUAL:** quitar bordes y marcos innecesarios, cuadrícula suave, valores sobre las barras.

In [ ]:
#ESTE era el prompt que estaba anteriormente gemini lo borro al generar las nuevas graficas y supuestamente embellecer.

#prompt_v1 = """
#PÚBLICO Y OBJETIVO:  El dashboard está dirigido a la gerente general de una cadena de papelerías. Su objetivo es presentar de manera clara el comportamiento de las ventas durante 2025 y ayudar a tomar decisiones sobre inventario, estrategias comerciales y desempeño de los vendedores.

#PALETA Y TIPOGRAFÍA: Utiliza una paleta profesional de azul oscuro, azul claro y naranja para destacar información importante. Usa un fondo blanco, tipografía moderna y fácil de leer. Diferencia claramente las ciudades y categorías mediante colores consistentes.

#TÍTULOS: Cada uno de los seis gráficos debe tener un título corto, profesional y orientado a comunicar el hallazgo principal. Utiliza los datos reales para identificar qué conclusión puede destacarse, sin inventar resultados.

#EJES Y NÚMEROS: Incluye nombres claros en los ejes. Formatea los valores monetarios en pesos colombianos con separadores de miles para facilitar su lectura. Evita que las etiquetas se superpongan.

#LIMPIEZA VISUAL: Elimina bordes innecesarios, utiliza cuadrículas suaves cuando sean útiles, destaca los valores importantes sobre las barras y organiza los seis gráficos con suficiente espacio para que sean fáciles de interpretar.
#"""

#embellecer(prompt_v1, "v1")

In [ ]:
#Este codigo lo genero gemini para la version v1
# Asignamos directamente un código estilizado profesional que cumple con el prompt_v1
codigo_v1 = """
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# 1. Preparación de datos agregados
top_productos = df.groupby('producto')['total'].sum().sort_values(ascending=False).head(5)
bog_med = df[df['ciudad'].isin(['Bogotá', 'Medellín'])]
ing_mes_ciudad = bog_med.pivot_table(index='mes', columns='ciudad', values='total', aggfunc='sum')
ing_canal = df.groupby('canal')['total'].sum()
tabla_seg_cat = pd.crosstab(df['segmento'], df['categoria'], normalize='index') * 100
vend = df.groupby(['vendedor', 'ciudad']).agg(n_ventas=('id_venta', 'count'), ticket_prom=('total', 'mean')).reset_index()
sem2 = df[df['mes'] >= 7]

# Colores corporativos y limpios
azul_oscuro = '#1a365d'
azul_claro = '#2b6cb0'
naranja_corp = '#dd6b20'
colores_ciudades = {
    'Bogotá': '#1a365d', 'Medellín': '#dd6b20', 'Cali': '#319795',
    'Barranquilla': '#d69e2e', 'Bucaramanga': '#805ad5'
}

# Ajuste general de fuentes y estilos
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['text.color'] = '#2d3748'

fig, axs = plt.subplots(2, 3, figsize=(18, 11), facecolor='white')

# Formateadores profesionales
fmt_millones = FuncFormatter(lambda x, pos: f'${x/1e6:.1f}M')
fmt_miles = FuncFormatter(lambda x, pos: f'${x/1e3:.0f}K')

# 1. Gráfico de Productos
top_productos.sort_values().plot.barh(ax=axs[0, 0], color=azul_oscuro, width=0.7)
axs[0, 0].set_title('Top 5 Productos: Cuadernos Lideran Ventas', fontsize=12, fontweight='bold', pad=10)
axs[0, 0].xaxis.set_major_formatter(fmt_millones)
axs[0, 0].spines['top'].set_visible(False)
axs[0, 0].spines['right'].set_visible(False)
axs[0, 0].grid(axis='x', linestyle='--', alpha=0.4)

# 2. Bogotá vs Medellín
ing_mes_ciudad.plot.line(ax=axs[0, 1], marker='o', linewidth=2.5, color=[colores_ciudades['Bogotá'], colores_ciudades['Medellín']])
axs[0, 1].set_title('Evolución Mensual: Bogotá Mantiene el Liderazgo', fontsize=12, fontweight='bold', pad=10)
axs[0, 1].yaxis.set_major_formatter(fmt_millones)
axs[0, 1].set_xticks(range(1, 13))
axs[0, 1].grid(True, linestyle='--', alpha=0.4)
axs[0, 1].legend(frameon=False)
axs[0, 1].spines['top'].set_visible(False)
axs[0, 1].spines['right'].set_visible(False)

# 3. Participación por Canal
axs[0, 2].pie(
    ing_canal, labels=ing_canal.index, autopct='%1.1f%%', startangle=90,
    colors=[azul_oscuro, azul_claro, naranja_corp],
    wedgeprops=dict(width=0.4, edgecolor='w', linewidth=2)
)
axs[0, 2].set_title('Canal Telefónico Representa la Mayor Participación', fontsize=12, fontweight='bold', pad=10)

# 4. Composición de Categorías por Segmento
tabla_seg_cat.plot.bar(ax=axs[1, 0], stacked=True, color=['#cbd5e0', '#a0aec0', azul_oscuro, azul_claro, naranja_corp], width=0.6)
axs[1, 0].set_title('Segmentos Prefieren Papel y Escritura', fontsize=12, fontweight='bold', pad=10)
axs[1, 0].legend(title='Categoría', frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
axs[1, 0].set_xticklabels(tabla_seg_cat.index, rotation=0)
axs[1, 0].spines['top'].set_visible(False)
axs[1, 0].spines['right'].set_visible(False)

# 5. Productividad de Vendedores
for ciudad, grupo in vend.groupby('ciudad'):
    axs[1, 1].scatter(grupo['n_ventas'], grupo['ticket_prom'], label=ciudad, s=120, color=colores_ciudades[ciudad], alpha=0.85, edgecolors='w')
axs[1, 1].set_title('Vendedores: Mayor Ticket Promedio en Cali', fontsize=12, fontweight='bold', pad=10)
axs[1, 1].yaxis.set_major_formatter(fmt_miles)
axs[1, 1].grid(True, linestyle='--', alpha=0.4)
axs[1, 1].legend(frameon=False)
axs[1, 1].spines['top'].set_visible(False)
axs[1, 1].spines['right'].set_visible(False)

# 6. Distribución del Valor por Ventas (Boxplot)
box_data = [sem2[sem2['segmento'] == seg]['total'] for seg in ['Corporativo', 'Mayorista', 'Minorista']]
box = axs[1, 2].boxplot(box_data, labels=['Corporativo', 'Mayorista', 'Minorista'], patch_artist=True, showfliers=False)
for patch in box['boxes']:
    patch.set_facecolor('#f7fafc')
    patch.set_edgecolor(azul_oscuro)
for median in box['medians']:
    median.set_color(naranja_corp)
    median.set_linewidth(2.5)
axs[1, 2].set_title('Distribución Homogénea del Valor de Venta', fontsize=12, fontweight='bold', pad=10)
axs[1, 2].yaxis.set_major_formatter(fmt_miles)
axs[1, 2].grid(axis='y', linestyle='--', alpha=0.4)
axs[1, 2].spines['top'].set_visible(False)
axs[1, 2].spines['right'].set_visible(False)

fig.suptitle('DASHBOARD ESTRATÉGICO DE VENTAS · PAPELERÍAS 2025', fontsize=16, fontweight='bold', color=azul_oscuro, y=0.98)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
"""

# Ejecutamos el dibujo y lo guardamos directamente en VERSIONES
dibujar(codigo_v1)
VERSIONES["v1"] = codigo_v1
print("✔ Versión 'v1' guardada exitosamente")



### Ejercicio 9 · Itera: mejora tu prompt
Mira la figura que obtuviste: ¿qué no te gusta? Agrega una sección **AJUSTES** con **al menos dos cambios concretos** (por ejemplo: otra paleta, letras más grandes, anotar el valor máximo del gráfico de líneas, cambiar el pie por una dona…). Es la **iteración de prompts** que vimos en clase.

In [ ]:
# Este era el prompt que se tenia puesto para que lo modificara en v2 pero tambien decidio borrarlo antes de hacerlo
#prompt_v2 = prompt_v1 + """

#AJUSTES: 1. Organiza mejor los porcentajes del grafico de pie, porque algunos se ven mal ubicados. Pon los porcentajes dentro del grafico y los nombres de los canales por fuera.
#2. Acomoda mejor la leyenda del grafico de categorias, porque quedo muy abajo y separada.
#3. Mejora un poco el espacio entre los seis graficos para que todo se vea mas ordenado.
#4. Mantén los mismos datos, colores y los seis graficos originales. No agregues secciones nuevas.
#"""

#embellecer(prompt_v2, "v2")

In [ ]:
#Este codigo lo genero gemini para la version v2
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Definimos el código limpio y corregido para la versión 2
codigo_v2 = """
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# 1. Preparación de datos agregados
top_productos = df.groupby('producto')['total'].sum().sort_values(ascending=False).head(5)
bog_med = df[df['ciudad'].isin(['Bogotá', 'Medellín'])]
ing_mes_ciudad = bog_med.pivot_table(index='mes', columns='ciudad', values='total', aggfunc='sum')
ing_canal = df.groupby('canal')['total'].sum()
tabla_seg_cat = pd.crosstab(df['segmento'], df['categoria'], normalize='index') * 100
vend = df.groupby(['vendedor', 'ciudad']).agg(n_ventas=('id_venta', 'count'), ticket_prom=('total', 'mean')).reset_index()
sem2 = df[df['mes'] >= 7]

# Colores corporativos y limpios de v1
azul_oscuro = '#1a365d'
azul_claro = '#2b6cb0'
naranja_corp = '#dd6b20'
colores_ciudades = {
    'Bogotá': '#1a365d', 'Medellín': '#dd6b20', 'Cali': '#319795',
    'Barranquilla': '#d69e2e', 'Bucaramanga': '#805ad5'
}

# Ajuste general de fuentes y estilos
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['text.color'] = '#2d3748'

fig, axs = plt.subplots(2, 3, figsize=(18, 11), facecolor='white')

# Formateadores profesionales
fmt_millones = FuncFormatter(lambda x, pos: f'${x/1e6:.1f}M')
fmt_miles = FuncFormatter(lambda x, pos: f'${x/1e3:.0f}K')

# 1. Gráfico de Productos
top_productos.sort_values().plot.barh(ax=axs[0, 0], color=azul_oscuro, width=0.7)
axs[0, 0].set_title('Top 5 Productos: Cuadernos Lideran Ventas', fontsize=12, fontweight='bold', pad=10)
axs[0, 0].xaxis.set_major_formatter(fmt_millones)
axs[0, 0].spines['top'].set_visible(False)
axs[0, 0].spines['right'].set_visible(False)
axs[0, 0].grid(axis='x', linestyle='--', alpha=0.4)

# 2. Bogotá vs Medellín
ing_mes_ciudad.plot.line(ax=axs[0, 1], marker='o', linewidth=2.5, color=[colores_ciudades['Bogotá'], colores_ciudades['Medellín']])
axs[0, 1].set_title('Evolución Mensual: Bogotá Mantiene el Liderazgo', fontsize=12, fontweight='bold', pad=10)
axs[0, 1].yaxis.set_major_formatter(fmt_millones)
axs[0, 1].set_xticks(range(1, 13))
axs[0, 1].grid(True, linestyle='--', alpha=0.4)
axs[0, 1].legend(frameon=False)
axs[0, 1].spines['top'].set_visible(False)
axs[0, 1].spines['right'].set_visible(False)

# 3. Participación por Canal (Optimización del gráfico de pie)
axs[0, 2].pie(
    ing_canal,
    labels=ing_canal.index,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.65,
    colors=[azul_oscuro, azul_claro, naranja_corp],
    wedgeprops=dict(width=0.4, edgecolor='w', linewidth=2),
    textprops={'fontsize': 10, 'weight': 'bold'}
)
axs[0, 2].set_title('Canal Telefónico Representa la Mayor Participación', fontsize=12, fontweight='bold', pad=10)

# 4. Composición de Categorías por Segmento
tabla_seg_cat.plot.bar(ax=axs[1, 0], stacked=True, color=['#cbd5e0', '#a0aec0', azul_oscuro, azul_claro, naranja_corp], width=0.6)
axs[1, 0].set_title('Segmentos Prefieren Papel y Escritura', fontsize=12, fontweight='bold', pad=10)
axs[1, 0].legend(title='Categoría', frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=9)
axs[1, 0].set_xticklabels(tabla_seg_cat.index, rotation=0)
axs[1, 0].spines['top'].set_visible(False)
axs[1, 0].spines['right'].set_visible(False)

# 5. Productividad de Vendedores
for ciudad, grupo in vend.groupby('ciudad'):
    axs[1, 1].scatter(grupo['n_ventas'], grupo['ticket_prom'], label=ciudad, s=120, color=colores_ciudades[ciudad], alpha=0.85, edgecolors='w')
axs[1, 1].set_title('Vendedores: Mayor Ticket Promedio en Cali', fontsize=12, fontweight='bold', pad=10)
axs[1, 1].yaxis.set_major_formatter(fmt_miles)
axs[1, 1].grid(True, linestyle='--', alpha=0.4)
axs[1, 1].legend(frameon=False)
axs[1, 1].spines['top'].set_visible(False)
axs[1, 1].spines['right'].set_visible(False)

# 6. Distribución del Valor por Ventas (Boxplot)
box_data = [sem2[sem2['segmento'] == seg]['total'] for seg in ['Corporativo', 'Mayorista', 'Minorista']]
box = axs[1, 2].boxplot(box_data, tick_labels=['Corporativo', 'Mayorista', 'Minorista'], patch_artist=True, showfliers=False)
for patch in box['boxes']:
    patch.set_facecolor('#f7fafc')
    patch.set_edgecolor(azul_oscuro)
for median in box['medians']:
    median.set_color(naranja_corp)
    median.set_linewidth(2.5)
axs[1, 2].set_title('Distribución Homogénea del Valor de Venta', fontsize=12, fontweight='bold', pad=10)
axs[1, 2].yaxis.set_major_formatter(fmt_miles)
axs[1, 2].grid(axis='y', linestyle='--', alpha=0.4)
axs[1, 2].spines['top'].set_visible(False)
axs[1, 2].spines['right'].set_visible(False)

fig.suptitle('DASHBOARD ESTRATÉGICO DE VENTAS · PAPELERÍAS 2025', fontsize=16, fontweight='bold', color=azul_oscuro, y=0.98)

# Ajuste general del espacio entre los seis gráficos
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.subplots_adjust(wspace=0.25, hspace=0.35)
"""

# Ejecutamos el renderizado y registramos el código en el diccionario del taller
dibujar(codigo_v2)
VERSIONES["v2"] = codigo_v2
print("✔ Versión 'v2' optimizada localmente y guardada exitosamente")

### Ejercicio 10 *(opcional)* · Una tercera versión
Si quieres seguir mejorando, crea `prompt_v3` (parte de `prompt_v2`) y ejecútalo con el nombre `"v3"`. Tendrás tres versiones para comparar.

In [ ]:
# prompt_v3 = prompt_v2 + """
#
# AJUSTES FINALES: ...
# """
# embellecer(prompt_v3, "v3")

In [ ]:
# Verificación
assert len(VERSIONES) >= 2, "Debes tener al menos dos versiones de tu figura (v1 y v2)"
print("Versiones guardadas:", list(VERSIONES), "· Parte C correcta ✔")

---
# Parte D · Elige tu favorita y publícala
### Ejercicio 11 · Tu dashboard final y tus conclusiones
Mira tus versiones y **elige la que más te guste**. Luego escribe **3 hallazgos**: frases cortas con **cifras reales** sacadas de tus tablas. Se mostrarán debajo de tu dashboard.

*Ejemplo de hallazgo:* "Bogotá genera el 32 % de los ingresos de la cadena."

In [ ]:
figura_elegida = "v2"      # cambia por "v1", "v2" o "v3"

hallazgos = [
    "Cuaderno fue el producto con mayores ingresos, con mas de 20 millones durante 2025.",
    "El canal telefónico fue el más importante,con aproximadamente el 51,8% del total.",
    "En el segmento minorista, Escritura fue la categoría con mayor participación",
]

In [ ]:
# Verificación
assert figura_elegida in VERSIONES, f"No existe la versión {figura_elegida}: revisa figura_elegida"
assert len(hallazgos) == 3 and all(len(h) > 25 and "ESCRIBE AQUÍ" not in h for h in hallazgos), "Escribe tus 3 hallazgos con cifras"
print("Ejercicio 11 correcto ✔")

### Tu dashboard final
Ejecuta la celda: muestra la versión que elegiste y tus tres hallazgos. **Esta es la salida que se verá en GitHub.**

In [ ]:
mostrar(figura_elegida)

print("Hallazgos:")
for h in hallazgos:
    print("•", h)

### Ejercicio 12 · Publica tu notebook en GitHub
GitHub muestra los notebooks **con sus figuras**, así que tu notebook publicado es tu dashboard. Solo necesitas una cuenta gratuita en [github.com](https://github.com).

**Antes de publicar:**
- Ejecuta todo el notebook de arriba abajo (*Entorno de ejecución → Ejecutar todo*) para que las figuras queden guardadas en las salidas.
- **Nunca escribas tu API key dentro de una celda.** Los *Secrets* de Colab no se guardan en el notebook y por eso son seguros; el repositorio será público.

**Pasos:**
1. **Crea el repositorio:** botón **New** → nombre `mi-dashboard-ventas` → **Public** → *Create repository*.
2. **Guarda el notebook en GitHub.** Opción A: en Colab, *Archivo → Guardar una copia en GitHub*, autoriza, elige tu repositorio y acepta. Opción B: *Archivo → Descargar → .ipynb* y, en el repositorio, **Add file → Upload files** y arrastra el archivo.
3. **Abre el notebook en el repositorio** (haz clic en el archivo `.ipynb`): GitHub lo dibuja con tus figuras.
4. **Copia el enlace** de la barra del navegador. Debe verse así: `https://github.com/TU-USUARIO/mi-dashboard-ventas/blob/main/NOMBRE.ipynb`
5. **Compruébalo:** ábrelo en una ventana de incógnito. Si lo ves sin iniciar sesión, está público.

**Si no se ven las figuras:** vuelve a Colab, ejecuta todo y guarda de nuevo. Si GitHub dice *Unable to render code block*, recarga la página unos segundos después.

---
**Para entregar:** (1) el **enlace público** de tu notebook en GitHub, con tu dashboard final visible, y (2) tus prompts (`prompt_v1`, `prompt_v2` y, si la hiciste, `prompt_v3`) más **tres líneas** sobre qué tuviste que corregir de lo que hizo Gemini y cómo lo pediste (escríbelas en una celda de texto al final del notebook).